# 🧠 Lesson 1 — Use Cases for Model Insights
### Kaggle Machine Learning Explainability
### Formato: Study + Reference + Hands‑on

Este notebook segue o estilo pedagógico definido para o curso:
- explicações conceituais
- mini‑referência
- código executável
- alinhamento total com a Lesson 1 do Kaggle

# 🎯 1. Objetivos da Lesson

Nesta lesson, você vai:

- entender **por que** precisamos de insights sobre modelos de Machine Learning  
- conhecer os **tipos de insights** que técnicas de explicabilidade podem fornecer  
- ver **casos de uso práticos** de insights de modelo  
- conectar explicabilidade com:
  - debugging  
  - feature engineering  
  - coleta de dados  
  - decisão humana  
  - confiança em modelos  

> Esta lesson é **conceitual**: ela prepara o terreno para as técnicas práticas (Permutation Importance, PDPs, SHAP).

# 📚 2. Glossário Técnico

- **Model Insight:** informação sobre como o modelo usa as features.  
- **Black Box:** modelo que prevê bem, mas não explica suas decisões.  
- **Debugging de Modelo:** encontrar erros a partir dos padrões aprendidos.  
- **Feature Engineering:** criação/transformação de variáveis.  
- **Explainability:** capacidade de interpretar modelos.  

# 🧾 3. Mini‑Referência (API Style)

A Lesson 1 é conceitual, mas ao longo do curso usaremos:

- `pandas.read_csv()` — carregar dados  
- `train_test_split()` — separar treino/validação  
- `RandomForestClassifier` — modelo base  
- `sklearn.metrics` — métricas  

Nesta lesson, o código é apenas de **setup**.

# 🧠 4. Introdução Conceitual

Modelos modernos são poderosos, mas muitas vezes funcionam como **caixas‑pretas**.

O curso de Explainability responde:

> “Como o modelo está tomando decisões?”

Isso é essencial para:
- evitar erros  
- melhorar modelos  
- tomar decisões melhores  
- ganhar confiança de stakeholders  

# 🧪 5. Implementação passo a passo (Setup mínimo)

Nesta lesson, vamos preparar o ambiente que será usado ao longo do curso. <br>
Como o dataset original do Kaggle Learn não está mais disponível, criamos
um **dataset sintético** com estrutura realista, adequado para as técnicas
de explicabilidade que estudaremos nas próximas lessons.

In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

print("Bibliotecas carregadas com sucesso.")

# Caminho raiz do projeto (um nível acima da pasta notebooks)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
print("PROJECT_ROOT =", PROJECT_ROOT)

Bibliotecas carregadas com sucesso.
PROJECT_ROOT = /home/moacir/ml/kaggle/explainability-ml


## 5.1. Gerando o dataset sintético

O objetivo é criar um dataset que represente um cenário realista de
readmissão hospitalar, com:

- variáveis numéricas (idade, tempo no hospital, exames, etc.)
- variáveis categóricas (gênero, raça, tipo de admissão)
- condições clínicas (diabetes, hipertensão)
- uma coluna alvo binária: **readmitted**

A lógica de geração mantém relações coerentes entre as variáveis,
permitindo que as técnicas de explicabilidade revelem padrões úteis.

In [4]:
np.random.seed(42)
N = 1200  # tamanho do dataset

# Variáveis numéricas
age = np.random.normal(55, 15, N).clip(18, 90)
num_procedures = np.random.poisson(2, N)
num_medications = np.random.poisson(5, N)
time_in_hospital = np.random.randint(1, 15, N)
lab_tests = np.random.normal(40, 10, N).clip(10, 80)

# Variáveis categóricas
gender = np.random.choice(["Male", "Female"], N)
race = np.random.choice(["White", "Black", "Hispanic", "Asian", "Other"], N)
admission_type = np.random.choice(
    ["Emergency", "Urgent", "Elective"],
    N,
    p=[0.6, 0.25, 0.15]
)
diabetes = np.random.choice(["Yes", "No"], N, p=[0.3, 0.7])
hypertension = np.random.choice(["Yes", "No"], N, p=[0.4, 0.6])

# Target: readmitted (com lógica realista)
logit = (
    0.015 * age +
    0.25 * time_in_hospital +
    0.12 * num_medications +
    0.8 * (diabetes == "Yes").astype(int) +
    0.6 * (hypertension == "Yes").astype(int) -
    12
)

prob = 1 / (1 + np.exp(-logit))
readmitted = np.random.binomial(1, prob)

# Construindo o DataFrame final
data = pd.DataFrame({
    "age": age,
    "gender": gender,
    "race": race,
    "admission_type": admission_type,
    "time_in_hospital": time_in_hospital,
    "num_procedures": num_procedures,
    "num_medications": num_medications,
    "lab_tests": lab_tests,
    "diabetes": diabetes,
    "hypertension": hypertension,
    "readmitted": readmitted
})

# Salvando no diretório padrão do projeto
RAW_DIR = os.path.join(PROJECT_ROOT, "data/raw")
os.makedirs(RAW_DIR, exist_ok=True)

DATA_PATH = os.path.join(RAW_DIR, "medical_readmissions.csv")
data.to_csv(DATA_PATH, index=False)

print("Dataset sintético criado com sucesso!")
print("Caminho:", DATA_PATH)
display(data.head())

Dataset sintético criado com sucesso!
Caminho: /home/moacir/ml/kaggle/explainability-ml/data/raw/medical_readmissions.csv


,age,gender,race,admission_type,time_in_hospital,num_procedures,num_medications,lab_tests,diabetes,hypertension,readmitted
0,62.450712,Female,Hispanic,Emergency,3,0,3,45.280360,No,Yes,0
1,52.926035,Female,White,Emergency,4,5,3,32.315645,Yes,No,0
2,64.715328,Female,Other,Emergency,4,1,4,40.490947,Yes,No,0
3,77.845448,Male,Asian,Urgent,6,3,4,26.482925,No,No,0
4,51.487699,Male,Black,Emergency,12,2,9,46.858905,Yes,No,0


## 5.2. Separando features e target (placeholder)

Ajuste o nome da coluna‑alvo quando o dataset estiver disponível.

In [5]:
TARGET_COL = "readmitted"  # exemplo

if 'data' in globals() and TARGET_COL in getattr(data, 'columns', []):
    X = data.drop(columns=[TARGET_COL])
    y = data[TARGET_COL]

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=0, stratify=y
    )

    print("Treino:", X_train.shape, "Validação:", X_valid.shape)
else:
    print("Dataset ainda não configurado para separação de X e y.")

Treino: (960, 10) Validação: (240, 10)


## 5.3. Codificando variáveis categóricas + Treinando um modelo base

 O RandomForest não aceita strings como entrada. <br>
 Portanto, precisamos transformar variáveis categóricas em números.

 Usamos One‑Hot Encoding com `pd.get_dummies`, exatamente como o Kaggle Learn faz.

In [7]:
# One-Hot Encoding
X_train_enc = pd.get_dummies(X_train)
X_valid_enc = pd.get_dummies(X_valid)

# Garantir que ambos tenham as mesmas colunas
X_train_enc, X_valid_enc = X_train_enc.align(X_valid_enc, join="left", axis=1, fill_value=0)

print("Shape após encoding:")
print("Treino:", X_train_enc.shape)
print("Validação:", X_valid_enc.shape)

# Treinando o modelo
model = RandomForestClassifier(
    n_estimators=100,
    random_state=0,
    n_jobs=-1
)

model.fit(X_train_enc, y_train)
print("Modelo treinado com sucesso!")

Shape após encoding:
Treino: (960, 19)
Validação: (240, 19)
Modelo treinado com sucesso!


# 🧩 6. Tipos de insights possíveis

O curso ensinará técnicas para responder:

1. **Quais features são mais importantes?**  
2. **Como cada feature influenciou uma previsão específica?**  
3. **Qual o efeito médio de cada feature nas previsões?**  

Técnicas:
- Permutation Importance  
- Partial Dependence Plots (PDPs)  
- SHAP Values  

# 🧪 7. Casos de uso práticos

- **7.1. Debugging** -> Insights revelam padrões estranhos, vazamentos e erros de pré‑processamento.
- **7.2. Feature Engineering** -> Insights mostram quais features importam e como interagem.
- **7.3. Coleta de dados** -> Empresas podem decidir **quais dados vale a pena coletar**.
- **7.4. Decisão humana** -> Em medicina, crédito e políticas públicas, explicações são essenciais.
- **7.5. Construir confiança** -> Stakeholders só confiam em modelos quando entendem suas decisões.

# 🧠 8. Observações pedagógicas do Copilot

- Esta lesson é **motivacional**, não técnica.  
- Ela prepara o terreno para Permutation Importance.  
- Conecta com o que você já sabe do curso anterior.  
- Explicabilidade é o próximo passo natural após modelagem.  

# ✅ 9. Conclusão

Nesta Lesson 1, você aprendeu:

- por que explicabilidade é necessária  
- quais tipos de insights são possíveis  
- como insights ajudam em debugging, feature engineering e confiança  

Próxima lesson:

> **Permutation Importance** — a primeira técnica prática de explicabilidade.